## Notebook 07: Production Offline Inference Testing & Edge-Case Validation

---

### 1. Architectural Role & Project Context
Once models are trained and validated, operational deployment requires an autonomous, zero-retraining inference engine. In field deployments (such as the autonomous water sampling boat and river monitoring stations), incoming telemetry arrives as heterogeneous JSON/dictionary payloads.

This notebook tests the production inference engine against real-world edge cases:
1. **Critical Contamination Event**: Acute industrial spill near a primary municipal water intake.
2. **LoRa Telemetry Blackout**: High hazard coupled with extended communication latency and stale sensor data.
3. **Partial Telemetry**: Missing optional inputs handled gracefully via automated imputation.
4. **Batch Inference**: High-throughput processing across multiple river basin monitoring nodes.
5. **Deterministic Repeatability**: Verification of invariant, reproducible outputs.

---

### 2. Linked Source Scripts (src/component04_ml/)
- **src/component04_ml/inference.py**:
  - DecisionInferenceEngine: Autonomous inference pipeline that loads models/selected_model.joblib and models/preprocessor.joblib.
  - predict_scenario(payload): Validates schema, preprocesses inputs, computes class probabilities, and attaches:
    - predicted_priority (P1-P4) & priority_description
    - 
easons: Natural language plain-English operational reasoning.
    - priority_content: SOP recommended actions, response time targets, and resource deployments.
    - operational_flags: Resource bottleneck, stale telemetry, and communication degradation indicators.
    - 	op_priority_drivers: Local SHAP explainability feature attributions.
  - predict_batch(df): High-throughput batch inference over DataFrames.
- **src/component04_ml/data_validation.py**:
  - DataValidator: Embedded within DecisionInferenceEngine to inspect incoming payloads for boundary violations.
- **src/component04_ml/explainability.py**:
  - DecisionModelExplainer: Embedded to compute real-time local driver features for every prediction.


In [3]:
import sys
from pathlib import Path
import pandas as pd
import json

root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.component04_ml.inference import DecisionInferenceEngine

engine = DecisionInferenceEngine()
print(f"Loaded primary inference engine: {engine.model_name}")


Loaded primary inference engine: XGBClassifier


### Section 1: Scenario 1 - Municipal Water Intake Contamination Crisis
**Edge Case:** High contamination probability ($0.94$) at `Kelani_Estuary_Municipal_Intake` with 45,000 exposed residents and online telemetry.
- **Expected Outcome:** Immediate **P1 (Critical)** classification with high probability and top SHAP drivers emphasizing intake criticality and population exposure.


In [4]:
scen_critical = {
    "scenario_id": "INTAKE_CRISIS_001",
    "catchment_zone": "Kelani_Estuary_Municipal_Intake",
    "contamination_probability": 0.94,
    "contamination_risk_category": "HIGH",
    "prediction_confidence": 0.96,
    "estimated_exposed_population": 45000,
    "infrastructure_importance": "CRITICAL",
    "communication_status": "ONLINE",
    "communication_delay_seconds": 15.0,
    "rainfall_mm": 60.0,
    "environmental_change_score": -0.55,
    "alternative_water_available": 0,
    "inspection_team_available": 1,
    "treatment_resource_available": 1,
    "sensor_data_freshness_seconds": 30.0,
    "missing_input_count": 0
}

res_1 = engine.predict_scenario(scen_critical)
print(json.dumps(res_1, indent=2))


{
  "scenario_id": "INTAKE_CRISIS_001",
  "predicted_priority": "P1",
  "priority_description": "Critical \u2014 Immediate Emergency Response Required",
  "reasons": [
    "Severe upstream contamination hazard detected with probability of 94.0%.",
    "Critical municipal drinking water intake or major facility directly threatened in affected zone.",
    "High population exposure with estimated 45,000 residents dependent on reach.",
    "AI Explainability (TreeSHAP): 'Infrastructure Hazard Index' is the leading driver escalating response priority."
  ],
  "priority_content": {
    "action_level": "EMERGENCY_INTERVENTION",
    "title": "Critical Disaster Response (P1)",
    "recommended_actions": [
      "Trigger immediate intake cutoff at municipal drinking water treatment facilities.",
      "Dispatch emergency incident response and rapid field water sampling teams.",
      "Issue high-priority public health alerts and activate alternative water supply bowsers.",
      "Switch telemetr

### Section 2: Scenario 2 - High Risk with LoRa Communication Blackout
**Edge Case:** Contamination probability ($0.78$) combined with severe communication latency ($14,400\text{s} = 4\text{h}$) and stale sensor readings ($28,800\text{s} = 8\text{h}$) in `DEGRADED` communication status.
- **Operational Logic:** Telemetry staleness penalties escalate uncertainty, triggering expedited response prioritization (P1/P2) to safeguard public health under uncertainty.


In [5]:
scen_comms_fail = {
    "scenario_id": "LORA_BLACKOUT_002",
    "catchment_zone": "Kelani_Upper_Industrial_Inflow",
    "contamination_probability": 0.78,
    "estimated_exposed_population": 12000,
    "infrastructure_importance": "MAJOR",
    "communication_status": "OFFLINE",
    "communication_delay_seconds": 3600.0,
    "sensor_data_freshness_seconds": 3700.0,
    "alternative_water_available": 1,
    "inspection_team_available": 0,
    "treatment_resource_available": 1
}

res_2 = engine.predict_scenario(scen_comms_fail)
print(f"Predicted Priority: {res_2['predicted_priority']}")
print("Operational Flags:", res_2["operational_flags"])


Predicted Priority: P1
Operational Flags: {'resource_bottleneck': False, 'stale_telemetry': True, 'communication_degraded': True}


### Section 3: Scenario 3 - Graceful Handling of Missing Optional Inputs
**Edge Case:** Incomplete telemetry payload where optional fields (`rainfall_mm`, `environmental_change_score`, resource flags) are absent.
- **Operational Logic:** `DecisionDataPreprocessor` safely imputes missing numerical values using training medians and missing indicators without crashing or throwing unhandled exceptions.


In [6]:
scen_missing = {
    "scenario_id": "PARTIAL_TELEMETRY_003",
    "catchment_zone": "Sedawatta_Floodplain_Basin",
    "contamination_probability": 0.52,
    "estimated_exposed_population": 6000,
    "infrastructure_importance": "STANDARD"
}

res_3 = engine.predict_scenario(scen_missing)
print(f"Predicted Priority: {res_3['predicted_priority']}")
print("Data Warnings:", res_3["data_quality_warnings"])


Predicted Priority: P2
Data Warnings: []


### Section 4: Scenario 4 - High-Throughput Batch Inference
**Source Method:** `DecisionInferenceEngine.predict_batch()`
- Simulates an operational river basin scan processing telemetry across multiple river reaches simultaneously.
- Demonstrates vectorized feature preprocessing and batch prediction throughput.


In [7]:
batch_df = pd.DataFrame([scen_critical, scen_comms_fail, scen_missing])
batch_results = engine.predict_batch(batch_df)

print(f"Processed {len(batch_results)} scenarios in batch mode:")
for r in batch_results:
    print(f"  {r['scenario_id']}: {r['predicted_priority']} (Probabilities: {r['class_probabilities']})")


Processed 3 scenarios in batch mode:
  INTAKE_CRISIS_001: P1 (Probabilities: {'P1': 0.9993, 'P2': 0.0005, 'P3': 0.0001, 'P4': 0.0})
  LORA_BLACKOUT_002: P1 (Probabilities: {'P1': 0.5171, 'P2': 0.4816, 'P3': 0.001, 'P4': 0.0003})
  PARTIAL_TELEMETRY_003: P2 (Probabilities: {'P1': 0.0003, 'P2': 0.8593, 'P3': 0.14, 'P4': 0.0004})


### Section 5: Scenario 5 - Deterministic Repeatability Verification
**Safety Verification Check:**
- Asserts that evaluating the identical input payload multiple times yields 100% bit-exact predicted priority levels and class probability vectors.
- Critical requirement for auditability, emergency post-mortems, and legal compliance.


In [8]:
r1 = engine.predict_scenario(scen_critical)
r2 = engine.predict_scenario(scen_critical)
assert r1["predicted_priority"] == r2["predicted_priority"], "Repeatability failure!"
assert r1["class_probabilities"] == r2["class_probabilities"], "Probability variance failure!"
print("Inference Repeatability Check: PASSED (100% deterministic identical outputs)")


Inference Repeatability Check: PASSED (100% deterministic identical outputs)
